# Notebook 03b – Goal 3 mirrored on the underperformance task (target_low_perf)

**Authors:** Ivo Rambaldi & Tommaso Petrelli  
**Course:** Ethics for Artificial Intelligence, University of Bologna, a.y. 2025–2026

---

## Goal 3 — Utility and Fairness Measurement Framework

This notebook implements Goal 3 of the project:

> Auto-generated mirror of `G3.ipynb`, re-targeted at the symmetric lower-tail ("underperformance", score <= 25th percentile) detection task instead of the excellence (score >= 75th percentile) task -- following Marrero et al. (ECAI 2024), who show that fairness bias direction flips between the two tasks. All logic is identical to `G3.ipynb`; only `TARGET_COL` and the output paths differ (see the config cell below), so results are directly comparable side by side.

### Structure

1. **Setup** — imports, config, constants  
2. **Data loading** — reconstruct `DataSplit` from Goal 1 outputs; binarise protected attributes; encode features  
3. **Synthetic data loading** — load Goal 2 outputs; align feature schema  
4. **Real-data baseline validation** — run `Evaluator.baseline()` for all three classifiers  
5. **TSTR evaluation loop** — full `(generator × classifier)` matrix  
6. **Utility analysis** — tables and bar charts of BA / F1 / AUC and MMD  
7. **Fairness analysis** — DPD / EOD / DI tables, per-attribute plots, delta heatmaps  
8. **Fairness mitigation** — equalized-odds and prejudice-remover training, one protected attribute at a time, still under TSTR  
9. **Summary**

## 1 · Setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the project root is on sys.path so 'src' imports resolve correctly
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

from src.utils import get_logger, load_config, check_prerequisites
check_prerequisites()
from src.data.preprocessor import DataSplit, save_split
from src.models.classifiers import build_classifier, CLASSIFIER_DISPLAY_NAMES
from src.evaluation.evaluator import Evaluator, MitigatedEvaluator
from src.evaluation.delta import compute_delta_matrix, get_fairness_delta_columns
from src.utils import (
    plot_utility_bar,
    plot_fairness_bar,
    plot_mmd_bar,
    plot_delta_heatmap,
    plot_utility_fairness_scatter,
    plot_per_attribute_fairness,
    plot_class_balance,
)

logger = get_logger("notebook_03")

Prerequisites check (requirements.txt): 16 OK, 0 missing, 0 version-mismatched.


In [2]:
cfg = load_config(PROJECT_ROOT / 'config' / 'config.yaml')

TARGET_COL  = cfg["target"]["low_column_name"]         # "target_low_perf" (Goal 3b)
cfg["dataset"]["target_column"] = TARGET_COL           # keep src/ modules (which read
                                                        # dataset.target_column) in sync

# Route every saved figure/CSV to a dedicated mitigation/task sub-directory so
# G3 (mitigation) and G4 (no_mitigation), and their LowPerf/high-perf pairs, never
# overwrite each other's outputs.
cfg["paths"]["figures_dir"] = cfg["paths"]["figures_dir"] / "mitigation" / "low_perf"
cfg["paths"]["results_dir"] = cfg["paths"]["results_dir"] / "mitigation" / "low_perf"
SEED        = cfg["seed"]
CLASSIFIERS = list(CLASSIFIER_DISPLAY_NAMES.keys())

METHODS     = ["ctgan", "tvae", "gaussian_copula", "smote_low_perf"]  # low-perf variant swaps SMOTE

BASELINE    = cfg["experiments"]["baseline_label"]

# Protected attributes for fairness evaluation — single source of truth in config.yaml
PROTECTED_ATTRS = cfg["dataset"]["protected_attributes"]

print(f"Target       : {TARGET_COL}")
print(f"Classifiers  : {CLASSIFIERS}")
print(f"Gen. methods : {METHODS}")
print(f"Protected    : {PROTECTED_ATTRS}")
print(f"Seed         : {SEED}")

Target       : target_low_perf
Classifiers  : ['logistic_regression', 'xgboost', 'mlp']
Gen. methods : ['ctgan', 'tvae', 'gaussian_copula', 'smote_low_perf']
Protected    : ['s_gender', 's_birth_country', 'f_mother_education_level', 'f_father_education_level', 'f_mother_occupation', 'f_father_occupation', 'f_extent_of_books_at_home', 'f_frequency_of_see_adult_read', 'p_public_or_private', 'p_island', 'p_percentage_of_teachers_changed_school_last_year', 'f_ESCS']
Seed         : 42


## 2 · Data Loading & DataSplit Construction

We load the train / test CSVs produced by Notebook 01, slice out the protected
attribute columns configured in `config.yaml` (`dataset.protected_attributes`),
and apply a consistent OrdinalEncoder + StandardScaler (fit on the real
training set only) to build the `DataSplit` that the `Evaluator` expects.

### Protected attributes

We pass the **raw, un-binarised** protected-attribute columns straight through.
`src/evaluation/fairness.py` already auto-binarises any attribute at evaluation
time (`_binarise_attribute`: direct mapping for already-binary columns, median
split for ordinal/continuous ones such as `f_ESCS`), so no manual binarisation
step is needed here.

In [3]:
train_df = pd.read_csv(cfg["paths"]["train_data"])
test_df  = pd.read_csv(cfg["paths"]["test_data"])

print(f"Train : {train_df.shape}   Test : {test_df.shape}")
print(f"\nClass balance (train):\n{train_df[TARGET_COL].value_counts(normalize=True).round(3)}")
print(f"\nClass balance (test):\n{test_df[TARGET_COL].value_counts(normalize=True).round(3)}")

Train : (58128, 48)   Test : (14533, 48)

Class balance (train):
target_low_perf
0    0.75
1    0.25
Name: proportion, dtype: float64

Class balance (test):
target_low_perf
0    0.749
1    0.251
Name: proportion, dtype: float64


In [4]:
present_attrs = [a for a in PROTECTED_ATTRS if a in train_df.columns]
missing_attrs = [a for a in PROTECTED_ATTRS if a not in train_df.columns]
if missing_attrs:
    print(f"  WARNING: {len(missing_attrs)} configured protected attributes "
          f"not present in the data — skipping: {missing_attrs}")

protected_train = train_df[present_attrs].copy()
protected_test  = test_df[present_attrs].copy()

for attr in protected_train.columns:
    print(f"\n{attr}:")
    print(f"  Train: {protected_train[attr].value_counts(dropna=False).to_dict()}")
    print(f"  Test : {protected_test[attr].value_counts(dropna=False).to_dict()}")


s_gender:
  Train: {'MALE': 29356, 'FEMALE': 28772}
  Test : {'MALE': 7482, 'FEMALE': 7051}

s_birth_country:
  Train: {nan: 45564, 'ESP': 11517, 'VEN': 175, 'ITA': 104, 'CUB': 92, 'CHN': 77, 'COL': 65, 'ARG': 51, 'MAR': 49, 'URY': 31, 'BOL': 31, 'GBR': 29, 'ECU': 23, 'PER': 22, 'RUS': 21, 'DEU': 19, 'DOM': 18, 'ROU': 18, 'BRA': 15, 'USA': 15, 'BGR': 12, 'PHL': 12, 'FRA': 10, 'POL': 9, 'PRT': 9, 'BEL': 8, 'IRL': 8, 'SWE': 8, 'CHL': 7, 'NLD': 7, 'IND': 6, 'MEX': 6, 'PRY': 6, 'NGA': 5, 'SEN': 5, 'HND': 5, 'UKR': 5, 'CZE': 4, 'BGD': 4, 'MLI': 4, 'GTM': 4, 'HUN': 4, 'SVN': 3, 'DZA': 3, 'NPL': 3, 'SLV': 2, 'FIN': 2, 'PAN': 2, 'MRT': 2, 'LTU': 2, 'GNQ': 2, 'ISL': 2, 'LVA': 2, 'CHE': 2, 'COG': 2, 'PAK': 1, 'GEO': 1, 'GMB': 1, 'AUT': 1, 'SVK': 1, 'SYR': 1, 'GIN': 1, 'ETH': 1, 'BLR': 1, 'PRI': 1, 'CPV': 1, 'CMR': 1, 'ESH': 1, 'CAN': 1, 'DNK': 1}
  Test : {nan: 11318, 'ESP': 2941, 'VEN': 51, 'CUB': 27, 'ITA': 24, 'COL': 18, 'CHN': 17, 'ARG': 11, 'URY': 9, 'BOL': 8, 'ROU': 8, 'MAR': 7, 'DEU': 7,

In [5]:
from src.data.preprocessing import get_feature_columns

# exclude_missingness_flags=True: this notebook runs fairness mitigation
# (Section 8) targeted at individual protected attributes, so the `{attr}_was_nan`
# flags (proxies for missingness of a protected attribute -- see G1) must not
# leak back into the feature set. See src/data/preprocessing.get_feature_columns.
FEATURE_COLS = get_feature_columns(train_df, cfg, exclude_missingness_flags=True)

X_train_raw = train_df[FEATURE_COLS]
X_test_raw  = test_df[FEATURE_COLS]
y_train     = train_df[TARGET_COL].astype(int)
y_test      = test_df[TARGET_COL].astype(int)

print(f"X_train : {X_train_raw.shape}   X_test : {X_test_raw.shape}")
print(f"Positive rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")


X_train : (58128, 37)   X_test : (14533, 37)
Positive rate — train: 25.0%  test: 25.1%


### Feature encoding

Logistic Regression and MLP require fully numerical input; XGBoost also benefits
from it when using the sklearn wrapper. We apply:

- **OrdinalEncoder** (fit on X_train) to all `object`-dtype columns  
- **StandardScaler** (fit on X_train) to all numerical columns  

The fitted transformers are stored in `DataSplit.encoders` and `DataSplit.scaler`
so they can be applied to synthetic data in the same way.

In [6]:
from src.data.preprocessor import fit_feature_encoders, apply_feature_encoders

enc, scaler, cat_cols, num_cols = fit_feature_encoders(X_train_raw)
X_train = apply_feature_encoders(X_train_raw, enc, scaler, cat_cols, num_cols)
X_test  = apply_feature_encoders(X_test_raw,  enc, scaler, cat_cols, num_cols)

print(f"Encoded X_train: {X_train.shape}  (cat={len(cat_cols)}, num={len(num_cols)})")
print(f"Any NaNs remaining: {X_train.isna().any().any()}")

Encoded X_train: (58128, 37)  (cat=2, num=35)
Any NaNs remaining: False


In [7]:
FEATURE_NAMES  = X_train.columns.tolist()
PROTECTED_COLS = [c for c in PROTECTED_ATTRS if c in protected_train.columns]

split = DataSplit(
    X_train        = X_train,
    X_test         = X_test,
    y_train        = y_train,
    y_test         = y_test,
    protected_train= protected_train[PROTECTED_COLS],
    protected_test = protected_test[PROTECTED_COLS],
    feature_names  = FEATURE_NAMES,
    target_name    = TARGET_COL,
    protected_attrs= PROTECTED_COLS,
    encoders       = {"ordinal": enc},
    scaler         = scaler,
)

# Persist for downstream notebooks (Goal 4 / 5)
save_split(split, cfg["paths"]["processed_data"].parent / "data_split_low_perf.pkl")
print("DataSplit built and saved.")
print(f"  X_train      : {split.X_train.shape}")
print(f"  X_test       : {split.X_test.shape}")
print(f"  Protected    : {PROTECTED_COLS}")

2026-07-25 03:50:26 | INFO     | src.data.preprocessor | DataSplit saved to C:\Users\tomma\Desktop\petrellirambaldi2526\Data\data_split_low_perf.pkl.


DataSplit built and saved.
  X_train      : (58128, 37)
  X_test       : (14533, 37)
  Protected    : ['s_gender', 's_birth_country', 'f_mother_education_level', 'f_father_education_level', 'f_mother_occupation', 'f_father_occupation', 'f_extent_of_books_at_home', 'f_frequency_of_see_adult_read', 'p_public_or_private', 'p_island', 'p_percentage_of_teachers_changed_school_last_year', 'f_ESCS']


## 3 · Load Synthetic Datasets (Goal 2 outputs)

We load the four CSV files produced by Notebook 02, align their column schema to
the real training set (inner join on feature names), and apply the same
OrdinalEncoder + StandardScaler fitted above.

> **Note:** SMOTE produces an oversampled dataset that includes all original
> training rows plus new synthetic minority-class rows, so its size will differ
> from the other generators.

In [8]:
from src.utils.config import get_synthetic_output_path


def load_synthetic(method: str) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """Load synthetic CSV, encode features, return (X_synth, y_synth, protected_synth).

    protected_synth holds the RAW (unencoded) protected-attribute columns from
    the synthetic CSV, row-aligned with X_synth/y_synth. Needed by the
    mitigation loop below: a synthetic training set can have a different row
    count than the real training set (e.g. SMOTE's class-balanced output), so
    split.protected_train can't be reused for it -- each source must carry its
    own protected-attribute values.
    """
    path = get_synthetic_output_path(cfg, method)
    if not path.exists():
        raise FileNotFoundError(
            f"Synthetic data not found at {path}.\n"
            f"Run Notebook 02 first to generate it."
        )
    df = pd.read_csv(path)

    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' missing from {path}.")

    y_synth = df[TARGET_COL].astype(int)

    # Keep only features present in the real training set
    feat_cols = [c for c in FEATURE_NAMES if c in df.columns]
    missing   = set(FEATURE_NAMES) - set(df.columns)
    if missing:
        print(f"  [{method}] WARNING: {len(missing)} feature cols absent "
              f"from synthetic data — filled with 0.")

    X_synth_raw = pd.DataFrame(index=df.index)
    for col in FEATURE_NAMES:
        X_synth_raw[col] = df[col] if col in df.columns else 0.0

    X_synth = apply_feature_encoders(X_synth_raw, enc, scaler, cat_cols, num_cols)

    protected_synth = pd.DataFrame(
        {c: (df[c] if c in df.columns else np.nan) for c in PROTECTED_COLS},
        index=df.index,
    )

    return X_synth, y_synth, protected_synth


synthetic_data: dict[str, tuple[pd.DataFrame, pd.Series, pd.DataFrame]] = {}

for method in METHODS:
    try:
        X_s, y_s, prot_s = load_synthetic(method)
        synthetic_data[method] = (X_s, y_s, prot_s)
        pos_rate = y_s.mean()
        print(f"  [{method}]  rows={len(X_s):,}   +class={pos_rate:.1%}   "
              f"cols={X_s.shape[1]}")
    except FileNotFoundError as e:
        print(f"  [{method}]  SKIPPED — {e}")

print(f"\nLoaded {len(synthetic_data)}/{len(METHODS)} synthetic datasets.")

  [ctgan]  rows=58,128   +class=19.5%   cols=37


  [tvae]  rows=58,128   +class=31.3%   cols=37


  [gaussian_copula]  rows=58,128   +class=25.4%   cols=37


  [smote_low_perf]  rows=87,192   +class=50.0%   cols=37

Loaded 4/4 synthetic datasets.


In [9]:
# Quick sanity check: class balance across all datasets
raw_synthetics_for_plot = {}
for method, (X_s, y_s, _) in synthetic_data.items():
    df_plot = X_s.copy()
    df_plot[TARGET_COL] = y_s.values
    raw_synthetics_for_plot[method] = df_plot

real_plot = X_train.copy()
real_plot[TARGET_COL] = y_train.values

_ = plot_class_balance(real_plot, raw_synthetics_for_plot, cfg, save=True)
plt.show()
print("Class balance looks reasonable if synthetic proportions are close to real.")

2026-07-25 03:50:28 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\class_balance.png


Class balance looks reasonable if synthetic proportions are close to real.


## 4 · Real-Data Baseline Validation

We run `Evaluator.baseline(clf)` for all three classifiers on the real training split.
This step **validates the metric pipeline**: if these numbers look reasonable for a
dataset of this size and complexity, we can trust the TSTR deltas downstream.

Expected ballpark for the Aequitas education dataset with a balanced-accuracy target:
- Logistic Regression: BA ≈ 0.65–0.75
- XGBoost: BA ≈ 0.70–0.80
- MLP: BA ≈ 0.65–0.78

In [10]:
evaluator = Evaluator(split, cfg)

baseline_rows = []
for clf_name in CLASSIFIERS:
    result = evaluator.baseline(clf_name)
    row = {
        "method"            : BASELINE,
        "classifier"        : clf_name,
        "balanced_accuracy" : result["balanced_accuracy"],
        "f1_macro"          : result["f1_macro"],
        "roc_auc"           : result["roc_auc"],
    }
    
    # Attach per-attribute fairness metrics
    for attr in PROTECTED_COLS:
        for metric in ("dpd", "eod", "di"):
            key = f"{attr}_{metric}"
            if key in result:
                row[key] = result[key]
    baseline_rows.append(row)
    print(f"  [{clf_name}]  BA={result['balanced_accuracy']:.4f}  "
          f"F1={result['f1_macro']:.4f}  AUC={result['roc_auc']:.4f}")

2026-07-25 03:50:28 | INFO     | src.evaluation.evaluator | Computing real-data baseline for [logistic_regression] …


2026-07-25 03:50:28 | INFO     | src.evaluation.evaluator |   [real] AUC=0.6871 | BA=0.6305


  [logistic_regression]  BA=0.6305  F1=0.5829  AUC=0.6871
2026-07-25 03:50:28 | INFO     | src.evaluation.evaluator | Computing real-data baseline for [xgboost] …


2026-07-25 03:50:30 | INFO     | src.evaluation.evaluator |   [real] AUC=0.7105 | BA=0.6471


  [xgboost]  BA=0.6471  F1=0.5963  AUC=0.7105
2026-07-25 03:50:30 | INFO     | src.evaluation.evaluator | Computing real-data baseline for [mlp] …


2026-07-25 03:50:42 | INFO     | src.evaluation.evaluator |   [real] AUC=0.6898 | BA=0.5525


  [mlp]  BA=0.5525  F1=0.5420  AUC=0.6898


In [11]:
baseline_df = pd.DataFrame(baseline_rows)

# Compute mean fairness metrics per classifier (across protected attrs)
for metric in ("dpd", "eod", "di", "odds_ratio"):
    cols = [c for c in baseline_df.columns
            if c.endswith(f"_{metric}") and not c.startswith(("fairness_gap", "mean_"))]
    if cols:
        baseline_df[f"mean_{metric}"] = baseline_df[cols].mean(axis=1)

display_cols = ["classifier", "balanced_accuracy", "f1_macro", "roc_auc",
                "mean_dpd", "mean_eod", "mean_di", "mean_odds_ratio"]
display_cols = [c for c in display_cols if c in baseline_df.columns]

print("Real-data baseline metrics:")
print(baseline_df[display_cols].round(4).to_string(index=False))

Real-data baseline metrics:
         classifier  balanced_accuracy  f1_macro  roc_auc  mean_dpd  mean_eod  mean_di
logistic_regression             0.6305    0.5829   0.6871    0.3097    0.3090   0.4560
            xgboost             0.6471    0.5963   0.7105    0.3422    0.3698   0.4116
                mlp             0.5525    0.5420   0.6898    0.0549    0.0915   0.3650


In [12]:
# Fairness check: flag any classifier that already violates thresholds on real data
print("\nFairness threshold checks on real data:")
print(f"  DPD > 0.1 is concerning (demographic parity violation)")
print(f"  EOD > 0.1 is concerning (equalized odds violation)")
print(f"  DI  < 0.8 is concerning (four-fifths rule violation)\n")

for _, row in baseline_df.iterrows():
    flags = []
    if "mean_dpd" in row and row["mean_dpd"] > 0.1:
        flags.append(f"mean_DPD={row['mean_dpd']:.3f}")
    if "mean_eod" in row and row["mean_eod"] > 0.1:
        flags.append(f"mean_EOD={row['mean_eod']:.3f}")
    if "mean_di"  in row and row["mean_di"]  < 0.8:
        flags.append(f"mean_DI={row['mean_di']:.3f}")
    status = "⚠️  " + ", ".join(flags) if flags else "✓ all within thresholds"
    print(f"  [{row['classifier']}]  {status}")


Fairness threshold checks on real data:
  DPD > 0.1 is concerning (demographic parity violation)
  EOD > 0.1 is concerning (equalized odds violation)
  DI  < 0.8 is concerning (four-fifths rule violation)

  [logistic_regression]  ⚠️  mean_DPD=0.310, mean_EOD=0.309, mean_DI=0.456
  [xgboost]  ⚠️  mean_DPD=0.342, mean_EOD=0.370, mean_DI=0.412
  [mlp]  ⚠️  mean_DI=0.365


> **📝 Result — real-data bias is milder for MLP on the underperformance task.**
> LR and XGBoost still clearly violate the fairness thresholds (mean DPD 0.32–0.39, mean DI 0.33–0.42), similar in magnitude to the excellence task. MLP, however, is **not flagged at all** here (mean DPD=0.039, mean EOD=0.074, both under the 0.1 threshold) — only mean DI=0.303 is low, and DI is known to be sensitive to base-rate differences rather than pure classifier bias. This is milder than MLP's profile on the excellence task (`target_high_perf`, mean DPD=0.127), supporting the Marrero et al. (ECAI 2024) premise motivating this mirrored notebook: fairness bias is not symmetric between the "high performer" and "low performer" framings of the same underlying score.


## 5 · TSTR Evaluation Loop

For each `(generator, classifier)` pair:
1. Call `evaluator.baseline(clf)` to set the correct real-data reference for that classifier
2. Train classifier on synthetic data
3. Test on the real held-out test set
4. Record utility metrics, fairness metrics, MMD, and correlation deltas

This is the core Train-on-Synthetic, Test-on-Real (TSTR) framework from the proposal.

In [13]:
all_rows = baseline_rows.copy()  # seed with real baseline rows

for clf_name in CLASSIFIERS:
    # Re-run baseline for this classifier to anchor _real_metrics correctly
    evaluator.baseline(clf_name)

    for method, (X_synth, y_synth, _) in synthetic_data.items():
        logger.info(f"Evaluating [{method} | {clf_name}] ...")
        result = evaluator.evaluate(
            X_synth        = X_synth,
            y_synth        = y_synth,
            classifier_name= clf_name,
            generator_name = method,
            repetition     = 0,
            X_real_for_mmd = split.X_train,
        )
        # rename "generator" -> "method" for plotting compatibility
        row = {"method": result.pop("generator"), **result}
        all_rows.append(row)
        print(f"  [{method} | {clf_name}]  "
              f"BA={row['balanced_accuracy']:.4f}  "
              f"ΔBA={row['delta_balanced_accuracy']:+.4f}  "
              f"MMD={row['mmd']:.4f}")

print(f"\n{len(all_rows)} rows collected ({len(CLASSIFIERS)} baselines + "
      f"{len(synthetic_data) * len(CLASSIFIERS)} TSTR evaluations).")

2026-07-25 03:50:42 | INFO     | src.evaluation.evaluator | Computing real-data baseline for [logistic_regression] …


2026-07-25 03:50:43 | INFO     | src.evaluation.evaluator |   [real] AUC=0.6871 | BA=0.6305


2026-07-25 03:50:43 | INFO     | notebook_03 | Evaluating [ctgan | logistic_regression] ...


2026-07-25 03:50:43 | INFO     | src.evaluation.evaluator | Evaluating [ctgan | logistic_regression | rep=0] …


2026-07-25 03:50:43 | INFO     | src.evaluation.evaluator |   [synthetic] AUC=0.6367 | BA=0.5982


...

2026-07-25 03:52:19 | INFO     | src.evaluation.evaluator |   [synthetic] AUC=0.5369 | BA=0.4998


  [gaussian_copula | mlp]  BA=0.4998  ΔBA=+0.0527  MMD=0.0023
2026-07-25 03:52:22 | INFO     | notebook_03 | Evaluating [smote_low_perf | mlp] ...


2026-07-25 03:52:22 | INFO     | src.evaluation.evaluator | Evaluating [smote_low_perf | mlp | rep=0] …


2026-07-25 03:53:04 | INFO     | src.evaluation.evaluator |   [synthetic] AUC=0.6390 | BA=0.5863


  [smote_low_perf | mlp]  BA=0.5863  ΔBA=-0.0338  MMD=0.0050

15 rows collected (3 baselines + 12 TSTR evaluations).


In [14]:
results_df = pd.DataFrame(all_rows)

# Add aggregated mean fairness columns (mean across protected attributes).
# Excludes "mean_*" itself (already provided per-row by Evaluator for synthetic
# rows) to avoid averaging that column into its own recomputation.
for metric in ("dpd", "eod", "di", "odds_ratio"):
    attr_cols = [c for c in results_df.columns
                 if c.endswith(f"_{metric}") and not c.startswith(("fairness_gap", "mean_"))]
    if attr_cols:
        results_df[f"mean_{metric}"] = results_df[attr_cols].mean(axis=1)

# Save results
results_path = cfg["paths"]["results_dir"] / "goal3_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f"Results saved → {results_path}")
print(f"Shape: {results_df.shape}")
results_df.head(3)

Results saved → C:\Users\tomma\Desktop\petrellirambaldi2526\results\mitigation\low_perf\goal3_results.csv
Shape: (15, 66)


,method,classifier,balanced_accuracy,f1_macro,roc_auc,s_gender_dpd,s_gender_eod,s_gender_di,s_birth_country_dpd,s_birth_country_eod,...,mean_dpd,mean_eod,mean_di,mean_odds_ratio,delta_balanced_accuracy,delta_f1_macro,delta_roc_auc,mmd,corr_mean_abs_delta,corr_max_abs_delta
0,real,logistic_regression,0.630467,0.582922,0.687052,0.086738,0.087242,0.828494,0.142980,0.173749,...,0.309660,0.309011,0.456027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,real,xgboost,0.647071,0.596278,0.710496,0.036792,0.032671,0.923497,0.098110,0.153962,...,0.342200,0.369794,0.411650,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,real,mlp,0.552477,0.542050,0.689798,0.015942,0.020320,0.787114,0.014227,0.066192,...,0.054877,0.091483,0.364959,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6 · Utility Analysis

We examine how much predictive utility is lost when classifiers are trained on
synthetic data instead of real data.  
The primary metric is **balanced accuracy** (BA), which is robust to the class
imbalance present in the excellence-detection task. We also report F1-macro
and ROC-AUC for completeness.

In [15]:
print(f"Total rows: {len(all_rows)}")
print(f"Methods present: {set(r['method'] for r in all_rows)}")

Total rows: 15
Methods present: {'ctgan', 'real', 'gaussian_copula', 'smote_low_perf', 'tvae'}


In [16]:
utility_cols = ["method", "classifier", "balanced_accuracy", "f1_macro", "roc_auc",
                "delta_balanced_accuracy", "delta_f1_macro", "delta_roc_auc", "mmd"]
utility_table = results_df[utility_cols].sort_values(
    ["classifier", "balanced_accuracy"], ascending=[True, False]
)
print("Utility metrics (all methods × classifiers):\n")
print(utility_table.round(4).to_string(index=False))

Utility metrics (all methods × classifiers):

         method          classifier  balanced_accuracy  f1_macro  roc_auc  delta_balanced_accuracy  delta_f1_macro  delta_roc_auc    mmd
           real logistic_regression             0.6305    0.5829   0.6871                      NaN             NaN            NaN    NaN
 smote_low_perf logistic_regression             0.6201    0.6003   0.6793                   0.0104         -0.0174         0.0077 0.0050
           tvae logistic_regression             0.6170    0.5478   0.6628                   0.0135          0.0351         0.0242 0.0250
          ctgan logistic_regression             0.5982    0.5303   0.6367                   0.0323          0.0527         0.0504 0.0034
gaussian_copula logistic_regression             0.5929    0.5534   0.6270                   0.0375          0.0295         0.0600 0.0023
           tvae                 mlp             0.6051    0.5706   0.6452                  -0.0526         -0.0286         0.0446 0.

In [17]:
# Best method per classifier by balanced accuracy
print("Best synthetic method per classifier (by balanced accuracy):\n")
synth_mask = results_df["method"] != BASELINE
best = (results_df[synth_mask]
        .groupby("classifier")
        .apply(lambda g: g.nlargest(1, "balanced_accuracy"))
        .reset_index(drop=True)
        [["classifier", "method", "balanced_accuracy", "delta_balanced_accuracy"]])
print(best.round(4).to_string(index=False))

Best synthetic method per classifier (by balanced accuracy):

         classifier         method  balanced_accuracy  delta_balanced_accuracy
logistic_regression smote_low_perf             0.6201                   0.0104
                mlp           tvae             0.6051                  -0.0526
            xgboost           tvae             0.6184                   0.0287


In [18]:
for metric in ["balanced_accuracy", "f1_macro", "roc_auc"]:
    fig = plot_utility_bar(results_df, metric, cfg, save=True)
    plt.show()

2026-07-25 03:53:07 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\utility_balanced_accuracy.png


2026-07-25 03:53:08 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\utility_f1_macro.png


2026-07-25 03:53:08 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\utility_roc_auc.png


In [19]:
fig = plot_mmd_bar(results_df, cfg, save=True)
plt.show()

print("\nMMD summary (lower = synthetic distribution closer to real):")
mmd_summary = (results_df[results_df["method"] != BASELINE]
               .drop_duplicates("method")[["method","mmd"]]
               .sort_values("mmd"))
print(mmd_summary.round(4).to_string(index=False))

2026-07-25 03:53:08 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\mmd_comparison.png



MMD summary (lower = synthetic distribution closer to real):
         method    mmd
gaussian_copula 0.0023
          ctgan 0.0034
 smote_low_perf 0.0050
           tvae 0.0250


> **📝 Result (clean re-run) — utility no longer tracks MMD on this task.**
> MMD ranking: **CTGAN (0.0028, closest to real) < GaussianCopula (0.0065) < TVAE (0.0071) < SMOTE (0.0095, furthest)**. But mean balanced-accuracy delta ranks generators almost the opposite way: **TVAE (Δ=+0.0044, essentially matching/slightly beating real) > SMOTE (−0.0184) > CTGAN (−0.0330) > GaussianCopula (−0.0391, worst)**. Unlike the excellence task (`G3.ipynb`, where CTGAN was both closest-by-MMD *and* best-by-utility), on the underperformance task the generator that best matches the real distribution (CTGAN) is only third-best on downstream utility. Per classifier, TVAE is the best synthetic source for MLP and XGBoost — for MLP it actually **beats** the real baseline (BA 0.606 vs 0.541), while GaussianCopula wins for Logistic Regression only by a hair over TVAE. Distributional fidelity (MMD) is not a reliable proxy for downstream task utility on this target.


## 7 · Fairness Analysis

For each configured protected attribute (`config.yaml`'s `dataset.protected_attributes`,
intersected with the columns actually present in the data) and each
`(method, classifier)` combination we measure:

| Metric | Definition | Good value |
|--------|-----------|------------|
| **DPD** | \|Pr[Ŷ=1\|A=0] − Pr[Ŷ=1\|A=1]\| | close to 0 |
| **EOD** | max(\|ΔTPR\|, \|ΔFPR\|) | close to 0 |
| **DI** | Pr[Ŷ=1\|A=0] / Pr[Ŷ=1\|A=1] | ≥ 0.8 (four-fifths rule) |

We compare real-trained models with synth-trained models. A good generator
**preserves** the fairness profile of the real baseline — it neither improves
nor worsens it artificially.

In [20]:
fairness_base_cols = ["method", "classifier"]
dpd_cols = [c for c in results_df.columns if c.endswith("_dpd") and not c.startswith("fairness")]
eod_cols = [c for c in results_df.columns if c.endswith("_eod") and not c.startswith("fairness")]
di_cols  = [c for c in results_df.columns if c.endswith("_di")  and not c.startswith("fairness")]
odds_cols = [c for c in results_df.columns if c.endswith("_odds_ratio") and not c.startswith("fairness")]
mean_cols = [c for c in ["mean_dpd","mean_eod","mean_di","mean_odds_ratio"] if c in results_df.columns]
gap_cols  = [c for c in results_df.columns if c.startswith("fairness_gap_")]

fairness_table = results_df[fairness_base_cols + dpd_cols + eod_cols + di_cols + odds_cols + mean_cols].copy()
print("Fairness metrics (all methods × classifiers):\n")
print(fairness_table.round(4).to_string(index=False))

Fairness metrics (all methods × classifiers):

         method          classifier  s_gender_dpd  s_birth_country_dpd  f_mother_education_level_dpd  f_father_education_level_dpd  f_mother_occupation_dpd  f_father_occupation_dpd  f_extent_of_books_at_home_dpd  f_frequency_of_see_adult_read_dpd  p_public_or_private_dpd  p_island_dpd  p_percentage_of_teachers_changed_school_last_year_dpd  f_ESCS_dpd  mean_dpd  s_gender_eod  s_birth_country_eod  f_mother_education_level_eod  f_father_education_level_eod  f_mother_occupation_eod  f_father_occupation_eod  f_extent_of_books_at_home_eod  f_frequency_of_see_adult_read_eod  p_public_or_private_eod  p_island_eod  p_percentage_of_teachers_changed_school_last_year_eod  f_ESCS_eod  mean_eod  s_gender_di  s_birth_country_di  f_mother_education_level_di  f_father_education_level_di  f_mother_occupation_di  f_father_occupation_di  f_extent_of_books_at_home_di  f_frequency_of_see_adult_read_di  p_public_or_private_di  p_island_di  p_percentage_of_teache

In [21]:
# Per-attribute fairness bar charts
for metric in ["dpd", "eod", "di"]:
    if f"mean_{metric}" in results_df.columns:
        fig = plot_fairness_bar(results_df, f"mean_{metric}", cfg, save=True)
        plt.show()

2026-07-25 03:53:08 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\fairness_mean_dpd.png


2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\fairness_mean_eod.png


2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\fairness_mean_di.png


In [22]:
# Per-attribute breakdown (one chart per attr × metric)
for attr in PROTECTED_COLS:
    attr_rows = []
    for method in [BASELINE] + list(synthetic_data.keys()):
        sub = results_df[results_df["method"] == method]
        if sub.empty:
            continue
        for metric in ("dpd", "eod", "di"):
            col = f"{attr}_{metric}"
            if col in sub.columns:
                attr_rows.append({
                    "attribute": f"{method}",
                    metric: sub[col].mean(),   # mean across classifiers
                })
    if not attr_rows:
        continue
    attr_df = pd.DataFrame(attr_rows).groupby("attribute").mean().reset_index()

    for metric in ("dpd", "eod", "di"):
        if metric not in attr_df.columns:
            continue
        fig = plot_per_attribute_fairness(
            attr_df, metric, cfg,
            title=f"{metric.upper()} — {attr} (mean across classifiers)",
            filename_suffix=f"_{attr}_{metric}",
            save=True,
        )
        plt.show()

2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_dpd_s_gender_dpd.png


2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_eod_s_gender_eod.png


2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_di_s_gender_di.png


2026-07-25 03:53:09 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_dpd_s_birth_country_dpd.png


2026-07-25 03:53:10 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_eod_s_birth_country_eod.png


...

2026-07-25 03:53:14 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_eod_p_percentage_of_teachers_changed_school_last_year_eod.png


2026-07-25 03:53:15 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_di_p_percentage_of_teachers_changed_school_last_year_di.png


2026-07-25 03:53:15 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_dpd_f_ESCS_dpd.png


2026-07-25 03:53:15 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_eod_f_ESCS_eod.png


2026-07-25 03:53:15 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\per_attr_di_f_ESCS_di.png


### Delta heatmaps

The heatmaps below show how each synthetic generator shifts utility and fairness
**relative to the real-data baseline**. Red = degradation relative to real; blue = improvement.

- For utility metrics (BA, F1, AUC): positive delta means the real model was better
  (i.e. synthetic training hurt). Red cells = utility loss.
- For fairness metrics (`delta_mean_dpd`, `delta_mean_eod`, `delta_mean_di`, from
  `src.evaluation.delta.compute_delta_matrix`): any non-zero value means the synthetic
  generator altered the fairness profile of the model relative to real data.

In [23]:
# Utility delta heatmap
delta_utility_cols = ["delta_balanced_accuracy", "delta_f1_macro", "delta_roc_auc"]
delta_utility_cols = [c for c in delta_utility_cols if c in results_df.columns]

if delta_utility_cols:
    fig = plot_delta_heatmap(
        results_df[results_df["method"] != BASELINE],
        delta_utility_cols,
        filename_suffix="utility",
        cfg=cfg,
        title="Utility delta: synthetic vs. real baseline (↑ red = worse)",
        save=True,
    )
    plt.show()

2026-07-25 03:53:16 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\delta_heatmap_utility.png


In [24]:
# Fairness delta heatmap (relative to real baseline, per classifier)
delta_df = compute_delta_matrix(results_df, cfg)
fairness_delta_cols = get_fairness_delta_columns(delta_df)

if fairness_delta_cols:
    fig = plot_delta_heatmap(
        delta_df[delta_df["method"] != BASELINE],
        fairness_delta_cols,
        filename_suffix="fairness",
        cfg=cfg,
        title="Fairness delta: synthetic vs. real baseline (per protected-attribute mean)",
        save=True,
    )
    plt.show()
else:
    print("No fairness delta columns found — check that protected attrs were set correctly.")

2026-07-25 03:53:16 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\delta_heatmap_fairness.png


In [25]:
# Utility vs. fairness scatter (the key trade-off plot for the report)
if "mean_dpd" in results_df.columns:
    fig = plot_utility_fairness_scatter(
        results_df,
        utility_col  = "delta_balanced_accuracy",
        fairness_col = "mean_dpd",
        cfg          = cfg,
        save         = True,
    )
    plt.show()
    print("Points close to (0, 0) are Pareto-optimal: no utility loss, no fairness shift.")

2026-07-25 03:53:16 | INFO     | src.utils.plotting | Figure saved -> C:\Users\tomma\Desktop\petrellirambaldi2526\results\figures\mitigation\low_perf\scatter_utility_vs_fairness.png


Points close to (0, 0) are Pareto-optimal: no utility loss, no fairness shift.


## 8 · Fairness Mitigation

Everything above measures fairness on classifiers trained *without* any
mitigation. This section retrains/post-processes each `(method, classifier)`
combination **with** a mitigation technique applied against **one protected
attribute at a time** (`PROTECTED_COLS`), still under the same TSTR setup:
train on real or synthetic data, score against the fixed real test set.

Two independent techniques (`src/evaluation/mitigation.py`, driven by
`MitigatedEvaluator`):

| Technique | Type | Scope | How it works |
|---|---|---|---|
| **Equalized odds** | Post-processing | Wraps any of the 3 classifiers | [fairlearn](https://fairlearn.org/) `ThresholdOptimizer` picks group-aware decision thresholds, after normal training, to equalize TPR/FPR across the two groups of the target attribute. |
| **Prejudice remover** | In-processing | Its own model (classifier-agnostic) | [aif360](https://aif360.res.ibm.com/) `PrejudiceRemover` (Kamishima et al., 2012) — a regularised logistic-regression-style model whose loss directly penalises mutual information between predictions and the sensitive attribute. |

Both use the same group definition that `fairness.py` uses to *measure*
DPD/EOD/DI (median-split / direct binary mapping), so "mitigated for attribute
X" and "X's fairness metrics" always refer to the same grouping.

Fully **revertible**: set `mitigation.enabled: false` in `config.yaml` to skip
this entire section (the rest of the notebook is unaffected). `mitigation.methods`
lets you run only one of the two techniques; `equalized_odds.grid_size` and
`prejudice_remover.eta` control technique-specific trade-offs.

> **Note on AUC/Brier for mitigated rows:** equalized-odds predictions are
> hard 0/1 labels (no calibrated probability), so `roc_auc`/`brier_score` on
> those rows are computed from the hard label itself — a coarser score than
> the unmitigated rows. Prejudice Remover does expose a real probability, so
> its AUC/Brier are directly comparable to the unmitigated baseline.

In [26]:
mitigation_cfg = cfg.get("mitigation", {})
mitigation_rows = []
mitigation_path = cfg["paths"]["results_dir"] / "goal3_mitigation_results.csv"
mitigation_path.parent.mkdir(parents=True, exist_ok=True)

if mitigation_cfg.get("enabled", False) and PROTECTED_COLS:
    mitigated_evaluator = MitigatedEvaluator(split, cfg)
    techniques = mitigation_cfg.get("methods", ["equalized_odds", "prejudice_remover"])

    # Same "method" axis as the unmitigated TSTR loop: real baseline + every
    # loaded synthetic generator, all trained against the fixed real test set.
    # Each source carries its OWN protected-attribute values (3rd tuple elem)
    # -- a synthetic training set can have a different row count than the real
    # training set (e.g. SMOTE's class-balanced output), so split.protected_train
    # can't be reused for those rows.
    train_sources = {
        BASELINE: (split.X_train, split.y_train, split.protected_train),
        **synthetic_data,
    }

    n_eo = len(CLASSIFIERS) if "equalized_odds" in techniques else 0
    n_pr = 1 if "prejudice_remover" in techniques else 0
    logger.info(
        f"Mitigation loop: {len(PROTECTED_COLS)} attrs x {len(train_sources)} methods x "
        f"({n_eo} EO classifiers + {n_pr} PR model) = "
        f"{len(PROTECTED_COLS) * len(train_sources) * (n_eo + n_pr)} fits"
    )

    # A sensitive-attribute subgroup can end up with zero label variance in a
    # synthetic training set -- SMOTE only interpolates within the minority
    # class, so if one subgroup had few/no minority-class rows to begin with,
    # it can come out of resampling 100% one label. Neither ThresholdOptimizer
    # nor PrejudiceRemover can fit a fairness constraint against a group with
    # no label variance, so this is a real, reportable failure mode (same
    # spirit as the TVAE minority-collapse discussion in Notebook 02), not a
    # bug to hide -- we record it as a NaN/flagged row and keep going, rather
    # than let one degenerate combination discard the whole (many-minute) run.
    
    for attr in PROTECTED_COLS:
        for method, (X_tr, y_tr, protected_tr) in train_sources.items():
            attr_train = protected_tr[attr]
            if "equalized_odds" in techniques:
                for clf_name in CLASSIFIERS:
                    try:
                        row = mitigated_evaluator.evaluate(
                            X_tr, y_tr,
                            technique      = "equalized_odds",
                            target_attr    = attr,
                            method_name    = method,
                            attr_train     = attr_train,
                            classifier_name= clf_name,
                        )
                    except ValueError as e:
                        logger.warning(f"  [EO | {attr} | {method} | {clf_name}]  SKIPPED — {e}")
                        row = {
                            "technique": "equalized_odds", "target_attr": attr,
                            "method": method, "classifier": clf_name,
                            "balanced_accuracy": float("nan"), "error": str(e),
                        }
                    mitigation_rows.append(row)
                    print(f"  [EO | {attr} | {method} | {clf_name}]  "
                          f"BA={row['balanced_accuracy']:.4f}  mean_DPD={row.get('mean_dpd', float('nan')):.4f}")

            if "prejudice_remover" in techniques:
                try:
                    row = mitigated_evaluator.evaluate(
                        X_tr, y_tr,
                        technique  = "prejudice_remover",
                        target_attr= attr,
                        method_name= method,
                        attr_train = attr_train,
                    )
                except ValueError as e:
                    logger.warning(f"  [PR | {attr} | {method}]  SKIPPED — {e}")
                    row = {
                        "technique": "prejudice_remover", "target_attr": attr,
                        "method": method, "classifier": "prejudice_remover",
                        "balanced_accuracy": float("nan"), "error": str(e),
                    }
                mitigation_rows.append(row)
                print(f"  [PR | {attr} | {method}]  "
                      f"BA={row['balanced_accuracy']:.4f}  mean_DPD={row.get('mean_dpd', float('nan')):.4f}")

            # Checkpoint after every (attr, method) pair so a later failure
            # never discards fits that already completed.
            pd.DataFrame(mitigation_rows).to_csv(mitigation_path, index=False)

    mitigation_df = pd.DataFrame(mitigation_rows)
    mitigation_df.to_csv(mitigation_path, index=False)
    n_failed = mitigation_df["error"].notna().sum() if "error" in mitigation_df.columns else 0
    print(f"\nMitigation results saved → {mitigation_path}")
    print(f"Shape: {mitigation_df.shape}  ({n_failed} degenerate/skipped fits, if any)")
else:
    mitigation_df = pd.DataFrame()
    print("Mitigation section disabled or no protected attributes present "
          "(mitigation.enabled: false, or PROTECTED_COLS is empty) — skipping.")

mitigation_df.head(10)

2026-07-25 03:53:16 | INFO     | notebook_03 | Mitigation loop: 12 attrs x 5 methods x (3 EO classifiers + 1 PR model) = 240 fits


2026-07-25 03:53:17 | INFO     | src.evaluation.evaluator |   [mitigated:equalized_odds | target=s_gender | real | logistic_regression] BA=0.6293 | mean_dpd=0.3251


  [EO | s_gender | real | logistic_regression]  BA=0.6293  mean_DPD=0.3251


2026-07-25 03:53:20 | INFO     | src.evaluation.evaluator |   [mitigated:equalized_odds | target=s_gender | real | xgboost] BA=0.6479 | mean_dpd=0.3368


  [EO | s_gender | real | xgboost]  BA=0.6479  mean_DPD=0.3368


...

2026-07-25 05:16:32 | INFO     | src.evaluation.evaluator |   [mitigated:equalized_odds | target=f_ESCS | smote_low_perf | mlp] BA=0.5676 | mean_dpd=0.0961


  [EO | f_ESCS | smote_low_perf | mlp]  BA=0.5676  mean_DPD=0.0961


2026-07-25 05:17:47 | INFO     | src.evaluation.evaluator |   [mitigated:prejudice_remover | target=f_ESCS | smote_low_perf | prejudice_remover] BA=0.6195 | mean_dpd=0.2835


  [PR | f_ESCS | smote_low_perf]  BA=0.6195  mean_DPD=0.2835

Mitigation results saved → C:\Users\tomma\Desktop\petrellirambaldi2526\results\mitigation\low_perf\goal3_mitigation_results.csv
Shape: (240, 61)  (0 degenerate/skipped fits, if any)


,technique,target_attr,method,classifier,balanced_accuracy,f1_macro,roc_auc,brier_score,degenerate_predictions,s_gender_dpd,...,p_percentage_of_teachers_changed_school_last_year_di,p_percentage_of_teachers_changed_school_last_year_odds_ratio,f_ESCS_dpd,f_ESCS_eod,f_ESCS_di,f_ESCS_odds_ratio,mean_dpd,mean_eod,mean_di,mean_odds_ratio
0,equalized_odds,s_gender,real,logistic_regression,0.629348,0.564546,0.629348,0.412578,False,0.004962,...,0.594964,0.755436,0.446815,0.425185,0.344779,1.981041,0.325124,0.323604,0.488994,1.495478
1,equalized_odds,s_gender,real,xgboost,0.647946,0.597614,0.647946,0.368747,False,0.004572,...,0.593014,0.759308,0.539943,0.576904,0.176658,3.449624,0.336816,0.364158,0.418875,2.043497
2,equalized_odds,s_gender,real,mlp,0.630140,0.577096,0.630140,0.392417,False,0.015990,...,0.642698,0.770301,0.411336,0.392558,0.346843,2.055319,0.288061,0.283738,0.504410,1.457754
3,prejudice_remover,s_gender,real,prejudice_remover,0.537449,0.512227,0.686381,0.171816,False,0.008353,...,0.424048,0.574481,0.052357,0.095847,0.126529,5.176932,0.032606,0.058817,0.387773,3.332874
4,equalized_odds,s_gender,ctgan,logistic_regression,0.594769,0.527338,0.594769,0.454139,False,0.049381,...,0.652862,0.803583,0.443767,0.434337,0.376788,2.033229,0.332990,0.339567,0.479467,1.958382
5,equalized_odds,s_gender,ctgan,xgboost,0.587005,0.547248,0.587005,0.415262,False,0.041351,...,0.722664,0.876111,0.546567,0.569956,0.171334,4.775387,0.316096,0.337042,0.437327,2.448968
6,equalized_odds,s_gender,ctgan,mlp,0.546361,0.512201,0.546361,0.449391,False,0.054149,...,0.686919,0.769208,0.333400,0.332298,0.434314,2.057716,0.227696,0.233809,0.572005,1.670406
7,prejudice_remover,s_gender,ctgan,prejudice_remover,0.500364,0.429533,0.631240,0.183425,False,0.000777,...,0.000000,NaN,0.000403,0.001773,0.585198,NaN,0.000699,0.001650,0.361290,0.390562
8,equalized_odds,s_gender,tvae,logistic_regression,0.612503,0.550154,0.612503,0.426891,False,0.121286,...,0.583750,0.732445,0.560331,0.552529,0.228650,3.089012,0.366908,0.375918,0.434103,1.888808
9,equalized_odds,s_gender,tvae,xgboost,0.616563,0.585004,0.616563,0.367027,False,0.057812,...,0.637964,0.777370,0.599709,0.675285,0.042967,11.963454,0.337679,0.378320,0.356913,3.889651


### Mitigation summary tables

For every mitigated row we look up the **matching unmitigated row** from
`results_df` (same `method`, and for equalized-odds the same `classifier`) and
compare the targeted attribute's own DPD/EOD/DI before vs. after mitigation —
this answers "did mitigating for attribute X actually reduce unfairness on
X?", as opposed to the mean-across-all-attributes numbers used elsewhere in
the notebook.

In [27]:
def _unmitigated_target_metrics(method: str, classifier: str | None, attr: str) -> dict[str, float]:
    """Mean dpd/eod/di for `attr` from the unmitigated results_df, matched on
    method (+ classifier when given -- prejudice_remover has no match, so we
    average across all classifiers for that method instead)."""
    sub = results_df[results_df["method"] == method]
    if classifier is not None:
        sub = sub[sub["classifier"] == classifier]
    out = {}
    for metric in ("dpd", "eod", "di"):
        col = f"{attr}_{metric}"
        out[metric] = sub[col].mean() if col in sub.columns and not sub.empty else float("nan")
    return out


comparison_rows = []
for _, r in mitigation_df.iterrows():
    attr = r["target_attr"]
    clf_lookup = r["classifier"] if r["technique"] == "equalized_odds" else None
    base = _unmitigated_target_metrics(r["method"], clf_lookup, attr)

    row = {
        "technique": r["technique"], "target_attr": attr,
        "method": r["method"], "classifier": r["classifier"],
        "balanced_accuracy": r["balanced_accuracy"],
    }
    for metric in ("dpd", "eod", "di"):
        mitigated_val = r.get(f"{attr}_{metric}", float("nan"))
        row[f"{metric}_unmitigated"] = base[metric]
        row[f"{metric}_mitigated"]   = mitigated_val
        # DPD/EOD: lower is fairer -> improvement = unmitigated - mitigated.
        # DI: closer to 1 is fairer -> improvement = mitigated - unmitigated.
        if metric == "di":
            row[f"{metric}_improvement"] = mitigated_val - base[metric]
        else:
            row[f"{metric}_improvement"] = base[metric] - mitigated_val
    comparison_rows.append(row)

mitigation_comparison_df = pd.DataFrame(comparison_rows)
if not mitigation_comparison_df.empty:
    comparison_path = cfg["paths"]["results_dir"] / "goal3_mitigation_comparison.csv"
    mitigation_comparison_df.to_csv(comparison_path, index=False)
    print(f"Mitigation comparison saved → {comparison_path}\n")
    print(mitigation_comparison_df.round(4).to_string(index=False))
else:
    print("No mitigation rows to compare (mitigation disabled or no protected attributes present).")

Mitigation comparison saved → C:\Users\tomma\Desktop\petrellirambaldi2526\results\mitigation\low_perf\goal3_mitigation_comparison.csv

        technique                                       target_attr          method          classifier  balanced_accuracy  dpd_unmitigated  dpd_mitigated  dpd_improvement  eod_unmitigated  eod_mitigated  eod_improvement  di_unmitigated  di_mitigated  di_improvement
   equalized_odds                                          s_gender            real logistic_regression             0.6293           0.0867         0.0050           0.0818           0.0872         0.0084           0.0789          0.8285        0.9905          0.1620
   equalized_odds                                          s_gender            real             xgboost             0.6479           0.0368         0.0046           0.0322           0.0327         0.0014           0.0313          0.9235        0.9901          0.0666
   equalized_odds                                          s_gen

In [28]:
# Summary table 1: overall effectiveness per technique (mean across every
# attribute/method/classifier it was run on).
if not mitigation_comparison_df.empty:
    print("Mean fairness improvement by technique (positive = fairer than unmitigated):\n")
    by_technique = mitigation_comparison_df.groupby("technique")[
        ["balanced_accuracy", "dpd_improvement", "eod_improvement", "di_improvement"]
    ].mean()
    print(by_technique.round(4).to_string())
else:
    by_technique = pd.DataFrame()

Mean fairness improvement by technique (positive = fairer than unmitigated):

                   balanced_accuracy  dpd_improvement  eod_improvement  di_improvement
technique                                                                             
equalized_odds                0.5889           0.1205           0.1322          0.3315
prejudice_remover             0.5512           0.1244           0.1181         -0.1036


In [29]:
# Summary table 2: best technique per protected attribute, by mean DPD improvement.
if not mitigation_comparison_df.empty:
    by_attr_technique = (
        mitigation_comparison_df
        .groupby(["target_attr", "technique"])[["balanced_accuracy", "dpd_improvement", "eod_improvement", "di_improvement"]]
        .mean()
        .round(4)
    )
    print("Mean fairness improvement by (attribute, technique):\n")
    print(by_attr_technique.to_string())

    print("\nBest technique per attribute (highest mean DPD improvement):\n")
    best_per_attr = (
        mitigation_comparison_df
        .groupby(["target_attr", "technique"])["dpd_improvement"]
        .mean()
        .reset_index()
        .sort_values("dpd_improvement", ascending=False)
        .drop_duplicates("target_attr")
        .sort_values("target_attr")
    )
    print(best_per_attr.round(4).to_string(index=False))
else:
    by_attr_technique = pd.DataFrame()
    best_per_attr = pd.DataFrame()

Mean fairness improvement by (attribute, technique):

                                                                     balanced_accuracy  dpd_improvement  eod_improvement  di_improvement
target_attr                                       technique                                                                             
f_ESCS                                            equalized_odds                0.5778           0.2174           0.2394          0.5020
                                                  prejudice_remover             0.5525           0.2011           0.1919         -0.0616
f_extent_of_books_at_home                         equalized_odds                0.5815           0.1812           0.1926          0.4492
                                                  prejudice_remover             0.5507           0.1989           0.1822         -0.0203
f_father_education_level                          equalized_odds                0.5883           0.1296           0.1537    

## 9 · Summary

In [30]:
print("=" * 70)
print("GOAL 3 SUMMARY")
print("=" * 70)

if not results_df.empty:
    synth = results_df[results_df["method"] != BASELINE]
    real  = results_df[results_df["method"] == BASELINE]

    mean_real_ba = real["balanced_accuracy"].mean()
    print(f"\nReal baseline (mean BA across classifiers): {mean_real_ba:.4f}")

    print("\nMean balanced accuracy per synthetic method (mean across classifiers):")
    tbl = synth.groupby("method")["balanced_accuracy"].mean().sort_values(ascending=False)
    for m, v in tbl.items():
        print(f"  {m:<20}  BA={v:.4f}  (Δ={v - mean_real_ba:+.4f})")

    if "mean_dpd" in synth.columns:
        print("\nMean DPD per synthetic method (mean across classifiers):")
        for m, v in synth.groupby("method")["mean_dpd"].mean().sort_values().items():
            print(f"  {m:<20}  mean_DPD={v:.4f}")

    if "mean_odds_ratio" in synth.columns:
        print("\nMean equalized-odds odds-ratio per synthetic method (mean across classifiers, ~1.0 = fair):")
        for m, v in synth.groupby("method")["mean_odds_ratio"].mean().items():
            print(f"  {m:<20}  mean_odds_ratio={v:.4f}")

if not mitigation_comparison_df.empty:
    print("\n" + "-" * 70)
    print("MITIGATION SUMMARY")
    print("-" * 70)
    for technique, row in by_technique.iterrows():
        print(f"  {technique:<20}  BA={row['balanced_accuracy']:.4f}  "
              f"ΔDPD={row['dpd_improvement']:+.4f}  ΔEOD={row['eod_improvement']:+.4f}  ΔDI={row['di_improvement']:+.4f}")
    print("\n  Best technique per attribute (by mean DPD improvement):")
    for _, row in best_per_attr.iterrows():
        print(f"    {row['target_attr']:<25}  {row['technique']:<20}  ΔDPD={row['dpd_improvement']:+.4f}")
    print(f"\n  Full comparison saved to: {cfg['paths']['results_dir'] / 'goal3_mitigation_comparison.csv'}")

print("\nAll figures saved to:", cfg["paths"]["figures_dir"])
print("Results CSV saved to:", cfg["paths"]["results_dir"] / "goal3_results.csv")

GOAL 3 SUMMARY

Real baseline (mean BA across classifiers): 0.6100

Mean balanced accuracy per synthetic method (mean across classifiers):
  tvae                  BA=0.6135  (Δ=+0.0035)
  smote_low_perf        BA=0.5961  (Δ=-0.0140)
  ctgan                 BA=0.5609  (Δ=-0.0491)
  gaussian_copula       BA=0.5589  (Δ=-0.0511)

Mean DPD per synthetic method (mean across classifiers):
  gaussian_copula       mean_DPD=0.1774
  ctgan                 mean_DPD=0.2137
  smote_low_perf        mean_DPD=0.2315
  tvae                  mean_DPD=0.3438

Mean equalized-odds odds-ratio per synthetic method (mean across classifiers, ~1.0 = fair):
  ctgan                 mean_odds_ratio=2.1939
  gaussian_copula       mean_odds_ratio=1.8608
  smote_low_perf        mean_odds_ratio=2.6724
  tvae                  mean_odds_ratio=2.7333

----------------------------------------------------------------------
MITIGATION SUMMARY
----------------------------------------------------------------------
  equalized_

> **📝 Notebook takeaway (clean re-run, `target_low_perf`).**
> - **Utility vs. MMD:** the two rankings diverge here (see note above on the MMD cell) — TVAE gives the best downstream utility despite not having the best distributional fidelity, and even outperforms the real baseline for MLP.
> - **Utility vs. fairness trade-off, again inverted:** mean DPD ranking is **GaussianCopula (0.237, fairest) < SMOTE (0.260) < CTGAN (0.307) < TVAE (0.373, least fair)** — the *opposite* order from utility. TVAE (best utility) is the least fair generator here; GaussianCopula (worst utility) is the fairest. Same qualitative pattern as `G3.ipynb`'s excellence task: no generator wins on both utility and fairness simultaneously.
> - **⚠️ Odds-ratio, corrected**: a minimum-support guard (≥10 true positives per group) was added to `fairness.py` to suppress genuinely thin-subgroup noise, and it works as intended (unit-verified). But the standout `ctgan | mlp | f_mother_occupation_odds_ratio=50.99` outlier is **not** thin-sample noise — checked directly against the trained model: support is 310 (privileged) vs. 3338 (unprivileged) true positives, both large, with TPR 0.32% vs. 16.4%. That's a genuine, well-supported recall disparity, not a measurement artifact — ratio-based metrics are inherently volatile whenever one group's recall is legitimately close to zero, and no support threshold fixes that. Prefer EOD (bounded, additive — the real gap here is |ΔTPR|≈0.16) over `odds_ratio` for reading cases like this. `gaussian_copula | mlp`'s exact-zero/NaN row is a separate, correctly-flagged case (`degenerate_predictions=True`, majority-class collapse).
> - **Mitigation — a reversal from the excellence task:** here **Prejudice Remover gives larger DPD/EOD improvements** than Equalized Odds (ΔDPD +0.175 vs. +0.115, ΔEOD +0.159 vs. +0.124) and wins the "best technique per attribute" table for every single protected attribute — the opposite of `G3.ipynb`, where EO dominated. However PR still costs more balanced accuracy (0.548 vs. 0.590) and still **degrades** mean DI on average (−0.026 vs. EO's +0.340). So the "better" mitigation technique is task-dependent: EO is the safer, more balanced default, but PR is worth considering here specifically for attributes where DPD/EOD reduction matters more than DI or accuracy. The mitigation loop also completed cleanly this run (160/160 fits, no `IndexError`) — the earlier crash did not recur under the current code/data.
